In [ ]:

# ================== 0) Mount & Imports ==================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, random, pickle
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from collections import Counter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)
np.random.seed(SEED)

GN_GROUPS = 32
def make_gn(C: int) -> nn.GroupNorm:
    g = min(GN_GROUPS, C)
    while g > 1 and (C % g) != 0:
        g //= 2
    return nn.GroupNorm(num_groups=max(1, g), num_channels=C)

# ================== 1) ResNet18 Backbone + Single Head ==================
def conv3x3(in_planes: int, out_planes: int, stride: int = 1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                     padding=1, bias=False)

class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes: int, planes: int, stride: int = 1):
        super().__init__()
        self.conv1 = conv3x3(in_planes, planes, stride)
        self.gn1 = make_gn(planes)
        self.conv2 = conv3x3(planes, planes)
        self.gn2 = make_gn(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes * self.expansion, kernel_size=1,
                          stride=stride, bias=False),
                make_gn(planes * self.expansion)
            )

    def forward(self, x):
        out = torch.relu(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        out = out + self.shortcut(x)
        out = torch.relu(out)
        return out

class ResNet18Backbone(nn.Module):
    def __init__(self, nf: int = 64):
        super().__init__()
        block = BasicBlock
        num_blocks = [2, 2, 2, 2]
        self.expansion = block.expansion
        self.nf = nf
        self.in_planes = nf

        self.conv1 = conv3x3(3, nf)
        self.gn1 = make_gn(nf)
        self.layer1 = self._make_layer(block, nf,     num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, nf * 2, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, nf * 4, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, nf * 8, num_blocks[3], stride=2)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        in_planes = self.in_planes
        for s in strides:
            layers.append(block(in_planes, planes, s))
            in_planes = planes * block.expansion
        self.in_planes = in_planes
        return nn.Sequential(*layers)

    def forward(self, x):
        out = torch.relu(self.gn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.avg_pool2d(out, out.shape[2])
        feat = out.view(out.size(0), -1)
        return feat

    @property
    def out_dim(self):
        return self.nf * 8 * self.expansion  # 512

class SingleHeadNet(nn.Module):
    def __init__(self, backbone: ResNet18Backbone, num_classes=6):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(backbone.out_dim, num_classes)

    def forward(self, x):
        feat = self.backbone(x)
        logits = self.head(feat)   # [B,6]
        return logits

# ================== 2) Paths ==================
BASE = "/content/drive/MyDrive/ML_Project/project_files/GridSearch_Mahalanobis"
os.makedirs(BASE, exist_ok=True)

MODEL_PATH = os.path.join(BASE, "finetuned_task3_best.pth")

SAVE_TOPK_PATH      = os.path.join(BASE, "CS_class4,5_topk.pkl")
SAVE_NEIGHBORS_PATH = os.path.join(BASE, "CS_neighbors_class4,5.pkl")

TOP_K = 100000
NUM_CS_SAMPLES = 2000
BATCH_SIZE = 32

# ================== 3) CIFAR-10 subset classes [4,5] remap -> {0,1} ==================
tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.4914, 0.4822, 0.4465),
                         std=(0.2470, 0.2435, 0.2616)),
])

test_full = datasets.CIFAR10(root="./data", train=False, download=True, transform=tf)
targets = test_full.targets

idx_45 = [i for i, t in enumerate(targets) if int(t) in [4, 5]]

class RemapCIFARSubset(Subset):
    def __init__(self, dataset, indices, keep_classes=(4, 5)):
        super().__init__(dataset, indices)
        orig_t = [dataset.targets[i] for i in indices]
        mapping = {c:i for i, c in enumerate(sorted(keep_classes))}  # 4->0, 5->1
        self.new_targets = [mapping[int(y)] for y in orig_t]

    def __getitem__(self, idx):
        x, _ = super().__getitem__(idx)
        return x, self.new_targets[idx]

test_45 = RemapCIFARSubset(test_full, idx_45, keep_classes=(4,5))

def get_random_subset(dataset, n_samples=2000, seed=42):
    rng = random.Random(seed)
    n = len(dataset)
    k = min(n_samples, n)
    idxs = rng.sample(range(n), k)
    return Subset(dataset, idxs)

subset = get_random_subset(test_45, n_samples=NUM_CS_SAMPLES, seed=SEED)
subset_loader = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# ✅ Quick sanity check for labels coming from subset_loader
cnt = Counter()
for _, y in subset_loader:
    cnt.update(y.tolist())

print("[CHECK] subset_loader label distribution:", dict(sorted(cnt.items())))
print("[CHECK] unique labels:", sorted(cnt.keys()))

# ================== 4) Tools (ravel/unravel) ==================
def unravel_index(index, shape):
    dims = []
    for s in reversed(shape):
        dims.append(index % s)
        index //= s
    return tuple(reversed(dims))

def ravel_index(multi_idx, shape):
    flat = 0
    for idx, dim in zip(multi_idx, shape):
        flat = flat * dim + idx
    return flat

# ================== 5) Confidence Sensitivity computation ==================
def compute_confidence_sensitivity_task45(model, dataloader, device):

    model.eval()

    cs = {
        n: torch.zeros_like(p, device=device)
        for n, p in model.named_parameters()
        if p.requires_grad and (not n.startswith("head"))
    }

    total_samples = 0

    for inputs, _ in dataloader:
        inputs = inputs.to(device)

        for ex in inputs:
            ex = ex.unsqueeze(0)   # [1,3,32,32]

            model.zero_grad(set_to_none=True)

            logits = model(ex)          # [1,6]
            logits_new = logits[:, 4:6] # [1,2] -> classes 4,5

            pred = torch.argmax(logits_new, dim=1)   # [1]
            fx = logits_new.gather(1, pred.view(1, 1)).squeeze()  # scalar

            fx.backward()

            for name, p in model.named_parameters():
                if name.startswith("head"):
                    continue
                if p.grad is not None:
                    cs[name] += (p.grad.detach() ** 2)

            total_samples += 1

    for name in cs:
        cs[name] /= max(1, total_samples)

    return cs

# ================== 6) Top-K & Neighbors ==================
def get_topk_cs_weights(cs_dict, model_state_dict, k):
    if k <= 0:
        return []

    all_entries = []
    for name, cs_tensor in cs_dict.items():
        flat_cs = cs_tensor.flatten()
        flat_w = model_state_dict[name].flatten()
        for i in range(flat_cs.numel()):
            all_entries.append({
                "name": name,
                "index": i,
                "value": float(flat_w[i].item()),
                "cs": float(flat_cs[i].item())
            })

    all_entries.sort(key=lambda x: x["cs"], reverse=True)
    return all_entries[:k]

def extract_conv_neighbors(topk_entries, model, cs_dict):
    neighbors = []
    if not topk_entries:
        return neighbors

    param_shapes = {name: p.shape for name, p in model.named_parameters()}
    cs_flat = {name: tens.flatten() for name, tens in cs_dict.items()}

    neighbor_offsets = [
        (0,0,-1,-1),(0,0,-1,1),(0,0,1,-1),(0,0,1,1),
        (0,0,-1,0),(0,0,1,0),(0,0,0,-1),(0,0,0,1),
    ]

    topk_set = set((e["name"], e["index"]) for e in topk_entries)
    added = set()

    for e in topk_entries:
        name, flat_idx = e["name"], e["index"]
        shape = param_shapes[name]

        if len(shape) != 4:
            continue

        oc, ic, kh, kw = unravel_index(flat_idx, shape)

        for do, di, dh, dw in neighbor_offsets:
            no, ni, nh, nw = oc + do, ic + di, kh + dh, kw + dw
            if 0 <= no < shape[0] and 0 <= ni < shape[1] and 0 <= nh < shape[2] and 0 <= nw < shape[3]:
                n_flat = ravel_index((no, ni, nh, nw), shape)
                key = (name, n_flat)
                if key in topk_set or key in added:
                    continue
                neighbors.append({
                    "name": name,
                    "index": n_flat,
                    "position": (no, ni, nh, nw),
                    "cs": float(cs_flat[name][n_flat].item())
                })
                added.add(key)

    return neighbors

# ================== 7) Build model & Load checkpoint ==================
model = SingleHeadNet(backbone=ResNet18Backbone(nf=64), num_classes=6).to(DEVICE)

ckpt = torch.load(MODEL_PATH, map_location=DEVICE)
state = ckpt["state"] if isinstance(ckpt, dict) and "state" in ckpt else ckpt

model.load_state_dict(state, strict=True)
print(f"[INFO] Loaded checkpoint from {MODEL_PATH}")

# ================== 8) Confidence Sensitivity computation ==================
cs_info = compute_confidence_sensitivity_task45(model, subset_loader, DEVICE)

topk_info = get_topk_cs_weights(cs_info, model.state_dict(), TOP_K)
neighbors_info = extract_conv_neighbors(topk_info, model, cs_info)

with open(SAVE_TOPK_PATH, "wb") as f:
    pickle.dump(topk_info, f)
with open(SAVE_NEIGHBORS_PATH, "wb") as f:
    pickle.dump(neighbors_info, f)

# ================== 9) Quick Stats ==================
print(f"✅ Confidence Sensitivity computed from {len(subset)} samples (Task: classes 4,5).")
print(f"✅ Top-K total: {len(topk_info)}")
conv_topk = [e for e in topk_info if len(model.state_dict()[e['name']].shape) == 4] if topk_info else []
print(f"📌 Top-K from Conv2D: {len(conv_topk)} ({100*len(conv_topk)/len(topk_info):.2f}% if any)")
print(f"📌 Neighbors extracted: {len(neighbors_info)}")
print("✔️ Done.")

Mounted at /content/drive


100%|██████████| 170M/170M [00:01<00:00, 94.6MB/s]


[CHECK] subset_loader label distribution: {4: 1000, 5: 1000}
[CHECK] unique labels: [4, 5]
[INFO] Loaded checkpoint from /content/drive/MyDrive/ML_Project/project_files/GridSearch_Mahalanobis/finetuned_task3_best.pth
✅ Confidence Sensitivity computed from 2000 samples (Task: classes 4,5).
✅ Top-K total: 100000
📌 Top-K from Conv2D: 94418 (94.42% if any)
📌 Neighbors extracted: 34187
✔️ Done.
